# 💼 AI Career Coach
### A Multi-Agent AI System · 100% local open-source LLMs (Qwen2.5-Instruct) · no API keys

---

## 1. Project Introduction

**Problem.** Tailoring a resume and a cover letter to a specific job, and honestly judging how well you fit it, is slow and hard to do objectively on your own.

**Solution.** A team of cooperating AI agents that takes a resume PDF and a job description and automatically produces: a structured resume profile, a job-fit score with matching/missing skills, concrete improvement feedback, and a tailored cover letter.

**Objectives**
- Data retrieval: read and extract text from a resume PDF.
- LLM analysis with a **local** Qwen model (no cloud API, no API key).
- Four agents that genuinely pass data to each other.
- Workflow orchestration, automated report output, error handling.

**Why multi-agent?** Each agent has one responsibility and its own prompt, input and output. Splitting the job makes each LLM call smaller and more reliable (important for small local models), makes errors easy to locate, and the output of one agent becomes the input of the next (Resume → Job-Fit → Feedback / Cover Letter), coordinated by an Orchestrator.

## 2. System Architecture

```
              ┌──────────────────────┐
              │ Resume PDF + Job Description  │
              └─────────┬───────────────┘
                        ▼
              ┌──────────────────┐
              │   Resume Agent   │  PDF → text → structured profile
              └────────┬───────────┘
                        │  profile (skills, experience, education)
                        ▼
              ┌──────────────────┐
              │  Job-Fit Agent   │  profile + job → fit score, gaps
              └────────┬───────────┘
                        │  fit score + matching/missing skills
            ┌─────────┴──────────┐
            ▼                      ▼
  ┌─────────────┐        ┌────────────────┐
  │ Feedback Agent │        │ Cover Letter Agent │
  └───────────┘        └───────────────┘
            │                      │
            └─────────┬──────────┘
                        ▼
              ┌──────────────────┐
              │   Orchestrator   │  runs agents in order, collects results
              └────────┬───────────┘
                        ▼
              ┌──────────────────┐
              │   career_report.md │
              └──────────────────┘

   Every agent calls the SAME  LLMManager  →  one local Qwen model
```

| Component | Responsibility |
|---|---|
| `LLMManager` | Loads the local model, generates text, safe JSON generation with retry |
| `ResumeAgent` | PDF → text → structured profile (name, skills, experience, education, summary) |
| `JobFitAgent` | Profile + job description → fit score, matching skills, missing skills |
| `FeedbackAgent` | Profile + fit result → strengths, improvements, skills to learn |
| `CoverLetterAgent` | Profile + job description + fit result → tailored cover letter (free text) |
| `Orchestrator` | Runs the agents in order, passes outputs along, handles errors, writes the report |

## 3. Environment Setup
No API keys needed — only open-source packages and model weights from the Hugging Face Hub (public models, no login).

In [1]:
!pip install -q transformers accelerate bitsandbytes pypdf streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 44.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 88.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 85.3 MB/s eta 0:00:00:00:010:01


## 4. Hardware / Runtime Detection
Picks model sizes that fit the available hardware, same approach as the course's local-model notebooks.

| Runtime | Model |
|---|---|
| GPU ≥ 20 GB | Qwen2.5-7B-Instruct (4-bit) |
| GPU < 20 GB (free T4) | Qwen2.5-1.5B-Instruct (fp16) |
| CPU only (slow) | Qwen2.5-1.5B-Instruct |

In [2]:
# @core:imports
import os, re, json, gc, time, math
from typing import Any, Dict, List, Optional

import torch
from pypdf import PdfReader
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [3]:
# @core:hardware
def detect_hardware() -> Dict[str, Any]:
    info = {"cuda_available": torch.cuda.is_available(), "device": "cpu", "gpu_name": None, "vram_gb": 0.0}
    if info["cuda_available"]:
        props = torch.cuda.get_device_properties(0)
        info.update(device="cuda", gpu_name=props.name, vram_gb=round(props.total_memory / 1024 ** 3, 1))
    return info


def select_models(hw: Dict[str, Any]):
    """Pick a LOCAL Qwen model that fits the hardware."""
    def spec(name, quant4=False):
        return {"name": name, "label": name.split("/")[-1], "quant4": quant4}

    if hw["device"] == "cuda" and hw["vram_gb"] >= 20:
        return spec("Qwen/Qwen2.5-7B-Instruct", quant4=True)
    return spec("Qwen/Qwen2.5-1.5B-Instruct")

In [4]:
HW = detect_hardware()
PRIMARY_SPEC = select_models(HW)

print("CUDA available :", HW["cuda_available"])
if HW["cuda_available"]:
    print(f"GPU            : {HW['gpu_name']}  ({HW['vram_gb']} GB VRAM)")
else:
    print("No GPU detected — the notebook will still run on CPU, but generation will be slow.")
print()
print("Local model:", PRIMARY_SPEC["name"], "(4-bit)" if PRIMARY_SPEC["quant4"] else "")

CUDA available : True
GPU            : Tesla T4  (14.6 GB VRAM)

Local model: Qwen/Qwen2.5-1.5B-Instruct 


## 5. Safe JSON Parsing
Strips code fences, trailing commas, and repairs JSON truncated by the token limit, before falling back to `None`.

In [5]:
# @core:json
def _close_truncated_json(s: str) -> str:
    stack, in_str, esc = [], False, False
    for ch in s:
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch in "{[":
                stack.append("}" if ch == "{" else "]")
            elif ch in "}]" and stack:
                stack.pop()
    if in_str:
        s += '"'
    s = re.sub(r",\s*$", "", s)
    return s + "".join(reversed(stack))


def extract_json(text: str) -> Optional[dict]:
    if not text:
        return None
    t = re.sub(r"```(?:json|JSON)?", "", text).strip()
    start = t.find("{")
    if start == -1:
        return None
    candidate = t[start:]
    end = candidate.rfind("}")
    versions = ([candidate[:end + 1]] if end != -1 else []) + [candidate]
    fixes = (
        lambda x: x,
        lambda x: re.sub(r",\s*([}\]])", r"\1", x),
        lambda x: _close_truncated_json(re.sub(r",\s*([}\]])", r"\1", x)),
    )
    for v in versions:
        for fix in fixes:
            try:
                obj = json.loads(fix(v), strict=False)
                if isinstance(obj, dict):
                    return obj
            except Exception:
                continue
    return None

## 6. Local LLM Manager

The single shared gateway every agent calls. `generate_json()` retries once with a different temperature if the model returns broken JSON, before giving up with a clear error.

In [6]:
# @core:llm
class LLMUnavailableError(RuntimeError):
    """Raised when the local model fails to load or generate a usable response."""


DEFAULT_SYSTEM = ("You are a precise career-coaching assistant. You only use information from the "
                  "resume and job description you are given. When asked for JSON you answer with "
                  "valid JSON only, without any extra commentary.")


class LLMManager:
    TEMPERATURES = (0.2, 0.6)  # attempt 1 / attempt 2 for JSON calls

    def __init__(self, spec: dict, max_new_tokens: int = 700, verbose: bool = True):
        self.spec = spec
        self.max_new_tokens = max_new_tokens
        self.verbose = verbose
        self.model = None
        self.tokenizer = None

    @property
    def current_model_name(self) -> str:
        return self.spec["label"]

    def status(self) -> dict:
        return {"model": self.spec["label"], "loaded": self.model is not None}

    def unload(self):
        if self.model is not None:
            del self.model
            self.model = None
        self.tokenizer = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if self.verbose:
            print("Model unloaded, GPU memory released.")

    def warm_up(self):
        if self.model is not None:
            return
        if self.verbose:
            print(f"Loading model: {self.spec['name']} ...")

        self.tokenizer = AutoTokenizer.from_pretrained(self.spec["name"], trust_remote_code=True)
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id

        kwargs = {"device_map": "auto", "trust_remote_code": True}
        if self.spec["quant4"]:
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16
            )
        else:
            kwargs["torch_dtype"] = torch.float16 if torch.cuda.is_available() else torch.float32

        self.model = AutoModelForCausalLM.from_pretrained(self.spec["name"], **kwargs)
        if self.verbose:
            print("Model ready.")

    def _raw_generate(self, prompt: str, system: str, max_new_tokens: int, do_sample: bool, temperature: float) -> str:
        messages = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=2048).to(self.model.device)

        gen_kwargs = dict(
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            eos_token_id=self.tokenizer.eos_token_id,
            pad_token_id=self.tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )
        if do_sample:
            gen_kwargs["temperature"] = temperature

        with torch.no_grad():
            outputs = self.model.generate(**inputs, **gen_kwargs)

        generated = outputs[0][inputs["input_ids"].shape[-1]:]
        return self.tokenizer.decode(generated, skip_special_tokens=True).strip()

    def generate(self, prompt: str, system: str = DEFAULT_SYSTEM, max_new_tokens: Optional[int] = None) -> str:
        self.warm_up()
        max_new_tokens = max_new_tokens or self.max_new_tokens
        try:
            return self._raw_generate(prompt, system, max_new_tokens, do_sample=True, temperature=0.7)
        except Exception as e:
            raise LLMUnavailableError(f"Local model failed to generate: {e}") from e

    def generate_json(self, prompt: str, system: str = DEFAULT_SYSTEM, max_new_tokens: int = 400,
                       normalizer=None) -> dict:
        self.warm_up()
        json_system = system + " Respond with ONLY a single valid JSON object, no markdown, no explanation."

        for temp in self.TEMPERATURES:
            try:
                raw = self._raw_generate(prompt, json_system, max_new_tokens, do_sample=True, temperature=temp)
            except Exception as e:
                raise LLMUnavailableError(f"Local model failed to generate: {e}") from e

            data = extract_json(raw)
            if data is not None and (normalizer is None or normalizer(data) is not None):
                return normalizer(data) if normalizer else data

        raise LLMUnavailableError("Could not get valid JSON from the model after two attempts.")

In [7]:
llm = LLMManager(PRIMARY_SPEC)
llm.warm_up()

Loading model: Qwen/Qwen2.5-1.5B-Instruct ...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model ready.


In [8]:
def show_status(manager: "LLMManager"):
    s = manager.status()
    print(f"Local LLM: {s['model']}")
    print(f"Loaded   : {'yes' if s['loaded'] else 'no'}")

show_status(llm)

Local LLM: Qwen2.5-1.5B-Instruct
Loaded   : yes


## 7. Resume PDF Processing

In [9]:
# @core:pdf_utils
class DocumentError(Exception):
    """A readable problem with the uploaded resume PDF."""


def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r" {2,}", " ", text)
    return text.strip()


def load_resume_text(pdf_path: str, max_chars: int = 15000) -> str:
    if not pdf_path or not os.path.exists(pdf_path):
        raise DocumentError("Resume PDF not found. Please upload a resume first.")

    try:
        reader = PdfReader(pdf_path)
        if reader.is_encrypted and not reader.decrypt(""):
            raise DocumentError("The PDF is password-protected. Please upload an unlocked copy.")
    except DocumentError:
        raise
    except Exception as e:
        raise DocumentError(f"Invalid or corrupted PDF ({type(e).__name__}). Please upload a different file.") from e

    if len(reader.pages) == 0:
        raise DocumentError("The PDF contains no pages.")

    text = ""
    for page in reader.pages:
        text += (page.extract_text() or "") + "\n"

    text = clean_text(text)
    if len(text) < 30:
        raise DocumentError("No readable text found — this PDF is probably a scanned image, not real text.")

    return text[:max_chars]

## 8. Resume Agent
**Input:** resume PDF path. **Output:** structured profile (`name`, `summary`, `skills`, `experience`, `education`).

In [10]:
# @core:resume_agent
def normalize_profile(data: dict) -> Optional[dict]:
    skills = [str(s).strip() for s in (data.get("skills") or []) if str(s).strip()]
    experience = []
    for e in (data.get("experience") or []):
        if isinstance(e, dict) and str(e.get("title", "")).strip():
            experience.append({
                "title": str(e.get("title", "")).strip(),
                "company": str(e.get("company", "")).strip(),
                "years": str(e.get("years", "")).strip(),
            })
    education = [str(ed).strip() for ed in (data.get("education") or []) if str(ed).strip()]
    summary = str(data.get("summary", "")).strip()
    name = str(data.get("name", "")).strip()

    if len(skills) < 1 or not summary:
        return None

    return {"name": name or "Unknown", "summary": summary, "skills": skills[:20],
            "experience": experience[:10], "education": education[:5]}


class ResumeAgent:
    name = "Resume Agent"

    def __init__(self, llm: "LLMManager"):
        self.llm = llm

    def run(self, pdf_path: str) -> Dict[str, Any]:
        text = load_resume_text(pdf_path)

        prompt = (
            "Extract a structured profile from this resume text.\n\n"
            f"RESUME TEXT:\n{text}\n\n"
            'Return ONLY JSON in this exact shape:\n'
            '{"name": "...", "summary": "one or two sentence professional summary", '
            '"skills": ["...", "..."], '
            '"experience": [{"title": "...", "company": "...", "years": "..."}], '
            '"education": ["..."]}'
        )

        return self.llm.generate_json(prompt, max_new_tokens=500, normalizer=normalize_profile)

## 9. Job-Fit Agent
**Input:** resume profile + job description. **Output:** `fit_score` (0-100), `matching_skills`, `missing_skills`, `notes`.

In [11]:
# @core:fit_agent
def normalize_fit(data: dict) -> Optional[dict]:
    try:
        score = int(float(data.get("fit_score", -1)))
    except (TypeError, ValueError):
        score = -1
    if not (0 <= score <= 100):
        return None

    matching = [str(s).strip() for s in (data.get("matching_skills") or []) if str(s).strip()]
    missing = [str(s).strip() for s in (data.get("missing_skills") or []) if str(s).strip()]
    notes = str(data.get("notes", "")).strip()

    return {"fit_score": score, "matching_skills": matching[:15], "missing_skills": missing[:15], "notes": notes}


class JobFitAgent:
    name = "Job-Fit Agent"

    def __init__(self, llm: "LLMManager"):
        self.llm = llm

    def run(self, profile: Dict[str, Any], job_description: str) -> Dict[str, Any]:
        if not job_description or not job_description.strip():
            raise DocumentError("Job description is empty. Please provide the job posting text.")

        prompt = (
            f"Candidate profile:\n{json.dumps(profile, indent=2)}\n\n"
            f"Job description:\n{job_description.strip()}\n\n"
            "Compare the candidate's skills and experience to the job description.\n"
            'Return ONLY JSON: {"fit_score": 0-100, "matching_skills": ["..."], '
            '"missing_skills": ["..."], "notes": "one short paragraph"}'
        )

        return self.llm.generate_json(prompt, max_new_tokens=400, normalizer=normalize_fit)

## 10. Feedback Agent
**Input:** profile + job-fit result. **Output:** `strengths`, `improvements`, `suggested_skills_to_learn`.

In [12]:
# @core:feedback_agent
def normalize_feedback(data: dict) -> Optional[dict]:
    strengths = [str(s).strip() for s in (data.get("strengths") or []) if str(s).strip()]
    improvements = [str(s).strip() for s in (data.get("improvements") or []) if str(s).strip()]
    skills_to_learn = [str(s).strip() for s in (data.get("suggested_skills_to_learn") or []) if str(s).strip()]

    if len(strengths) < 1 or len(improvements) < 1:
        return None

    return {"strengths": strengths[:8], "improvements": improvements[:8],
            "suggested_skills_to_learn": skills_to_learn[:8]}


class FeedbackAgent:
    name = "Feedback Agent"

    def __init__(self, llm: "LLMManager"):
        self.llm = llm

    def run(self, profile: Dict[str, Any], fit: Dict[str, Any]) -> Dict[str, Any]:
        prompt = (
            f"Candidate profile:\n{json.dumps(profile, indent=2)}\n\n"
            f"Job-fit analysis:\n{json.dumps(fit, indent=2)}\n\n"
            "Give the candidate honest, constructive feedback on their resume for this job.\n"
            'Return ONLY JSON: {"strengths": ["..."], "improvements": ["..."], '
            '"suggested_skills_to_learn": ["..."]}'
        )

        return self.llm.generate_json(prompt, max_new_tokens=400, normalizer=normalize_feedback)

## 11. Cover Letter Agent
**Input:** profile + job description + job-fit result. **Output:** a tailored cover letter (free text, not JSON).

In [13]:
# @core:letter_agent
class CoverLetterAgent:
    name = "Cover Letter Agent"

    def __init__(self, llm: "LLMManager"):
        self.llm = llm

    def run(self, profile: Dict[str, Any], job_description: str, fit: Dict[str, Any]) -> str:
        prompt = (
            f"Candidate profile:\n{json.dumps(profile, indent=2)}\n\n"
            f"Job description:\n{job_description.strip()}\n\n"
            f"Matching skills to emphasize: {', '.join(fit['matching_skills']) or 'the candidate\'s core skills'}\n\n"
            "Write a professional, concise cover letter (3-4 short paragraphs) for this candidate "
            "applying to this job. Return ONLY the letter text, no JSON, no extra notes."
        )

        system = ("You are a professional career-coaching writer. Write clear, confident, "
                  "human-sounding cover letters grounded only in the given profile.")

        return self.llm.generate(prompt, system=system, max_new_tokens=400)

## 12. Orchestrator
The only component that knows the whole workflow. Runs the four agents in order, passes each agent's output to the next, records timing/status, stops with a clear message if an agent fails, and writes `career_report.md`.

In [14]:
# @core:orchestrator
AGENT_NAMES = ["Resume Agent", "Job-Fit Agent", "Feedback Agent", "Cover Letter Agent"]


def build_report(result: dict) -> str:
    p, fit, fb = result["profile"], result["fit"], result["feedback"]
    skills = ", ".join(p["skills"]) or "none listed"
    matching = ", ".join(fit["matching_skills"]) or "none detected"
    missing = ", ".join(fit["missing_skills"]) or "none"
    strengths = "\n".join(f"- {s}" for s in fb["strengths"])
    improvements = "\n".join(f"- {s}" for s in fb["improvements"])
    learn = "\n".join(f"- {s}" for s in fb["suggested_skills_to_learn"]) or "- none suggested"
    trace = "\n".join(f"| {w['agent']} | {w['status']} | {w['seconds']} s |" for w in result["workflow"])

    return f"""# Career Coach Report

## 1. Candidate Profile
- **Name:** {p['name']}
- **Summary:** {p['summary']}
- **Skills:** {skills}

## 2. Job Fit
- **Fit score:** {fit['fit_score']} / 100
- **Matching skills:** {matching}
- **Missing skills:** {missing}
- **Notes:** {fit['notes']}

## 3. Feedback
**Strengths**
{strengths}

**Improvements**
{improvements}

**Skills worth learning**
{learn}

## 4. Cover Letter
{result['cover_letter']}

## 5. Workflow Trace
| Agent | Status | Time |
|---|---|---|
{trace}

*Model used: {result['llm_status']['model']}*
"""


class Orchestrator:
    def __init__(self, llm: "LLMManager"):
        self.llm = llm
        self.resume_agent = ResumeAgent(llm)
        self.fit_agent = JobFitAgent(llm)
        self.feedback_agent = FeedbackAgent(llm)
        self.letter_agent = CoverLetterAgent(llm)

    def run(self, pdf_path: str, job_description: str, report_path: str = "career_report.md") -> dict:
        workflow = []

        def step(agent_name, fn):
            print(f"\n[{agent_name}] running...")
            t0 = time.time()
            try:
                out = fn()
                workflow.append({"agent": agent_name, "status": "done", "seconds": round(time.time() - t0, 1)})
                print(f"[{agent_name}] done.")
                return out
            except (DocumentError, LLMUnavailableError) as e:
                workflow.append({"agent": agent_name, "status": f"failed: {e}", "seconds": round(time.time() - t0, 1)})
                print(f"[{agent_name}] failed: {e}")
                raise

        try:
            profile = step("Resume Agent", lambda: self.resume_agent.run(pdf_path))
            fit = step("Job-Fit Agent", lambda: self.fit_agent.run(profile, job_description))
            feedback = step("Feedback Agent", lambda: self.feedback_agent.run(profile, fit))
            cover_letter = step("Cover Letter Agent", lambda: self.letter_agent.run(profile, job_description, fit))
        except (DocumentError, LLMUnavailableError) as e:
            return {"status": "failed", "error": str(e), "workflow": workflow, "llm_status": self.llm.status()}

        result = {
            "status": "completed",
            "profile": profile,
            "fit": fit,
            "feedback": feedback,
            "cover_letter": cover_letter,
            "workflow": workflow,
            "llm_status": self.llm.status(),
        }
        result["report_markdown"] = build_report(result)
        with open(report_path, "w", encoding="utf-8") as f:
            f.write(result["report_markdown"])
        result["report_path"] = report_path

        return result

## 13. Resume Upload & Job Description
Run this cell and pick a resume PDF from your computer (Colab), or set `pdf_path` manually.

In [22]:
import glob

pdf_path = None

# --- Option A: Kaggle — look for a PDF already added via "+ Add Input" (right sidebar) ---
kaggle_pdfs = glob.glob("/kaggle/input/**/*.pdf", recursive=True)

if kaggle_pdfs:
    print("Found PDF(s) under /kaggle/input:")
    for f in kaggle_pdfs:
        print(" ", f)
    pdf_path = kaggle_pdfs[0]
    print(f"\nUsing: {pdf_path}  (edit pdf_path below if this picked the wrong file)")

# --- Option B: Google Colab — interactive upload widget ---
elif True:
    try:
        from google.colab import files
        uploaded = files.upload()
        pdfs = [name for name in uploaded if name.lower().endswith(".pdf")]
        if pdfs:
            pdf_path = pdfs[0]
            print(f"Using '{pdf_path}' ({os.path.getsize(pdf_path) / 1024:.0f} KB)")
        else:
            print("No PDF uploaded. Run this cell again and choose a .pdf file.")
    except ImportError:
        print("Not running in Google Colab, and no PDF found under /kaggle/input.")
        print("Kaggle: use the '+ Add Input' panel (right sidebar) to add your resume PDF, then rerun this cell.")
        print("Elsewhere: set pdf_path = '/path/to/resume.pdf' manually in the next line.")
    except Exception as e:
        # Covers the known Colab bug: "Upload widget is only available when the cell
        # has been executed in the current browser session."
        print(f"Upload widget failed: {e}")
        print("Fix: hard-refresh the Colab tab, reconnect the runtime, then click INTO this cell")
        print("and run it directly (the play button on the cell) rather than via Run All.")

# --- Option C: set manually (works everywhere, incl. local Jupyter) ---
pdf_path = "/kaggle/input/datasets/roqayamuhammad/roqayacv/Roqaya_Muhammad_cv_AI.pdf"   # Kaggle


if pdf_path:
    print(f"\npdf_path is set to: {pdf_path}")
else:
    print("\npdf_path is not set yet — set it manually above, then rerun this cell or continue.")

Found PDF(s) under /kaggle/input:
  /kaggle/input/datasets/roqayamuhammad/roqayacv/Roqaya_Muhammad_cv_AI.pdf

Using: /kaggle/input/datasets/roqayamuhammad/roqayacv/Roqaya_Muhammad_cv_AI.pdf  (edit pdf_path below if this picked the wrong file)

pdf_path is set to: /kaggle/input/datasets/roqayamuhammad/roqayacv/Roqaya_Muhammad_cv_AI.pdf


In [23]:
job_description = """
 Software Engineer, Python, 2+ years experience, REST APIs, SQL, cloud (AWS/GCP), team collaboration.
""".strip()

print(f"Job description set ({len(job_description)} characters).")

Job description set (148 characters).


## 14. Run Complete Workflow

In [25]:
if pdf_path is None:
    print("Please upload a resume PDF first (section 13).")
else:
    orchestrator = Orchestrator(llm)
    result = orchestrator.run(pdf_path, job_description)
    print("\nWorkflow status:", result["status"].upper())
    if result["status"] != "completed":
        print("Problem:", result["error"])


[Resume Agent] running...
[Resume Agent] done.

[Job-Fit Agent] running...
[Job-Fit Agent] done.

[Feedback Agent] running...
[Feedback Agent] done.

[Cover Letter Agent] running...
[Cover Letter Agent] done.

Workflow status: COMPLETED


## 15. Display Results

In [26]:
from IPython.display import Markdown, display

if "result" in globals() and result["status"] == "completed":
    display(Markdown(result["report_markdown"]))
    print()
    show_status(llm)
else:
    print("No completed result yet — run section 14 first.")

# Career Coach Report

## 1. Candidate Profile
- **Name:** Roqaya Muhammad
- **Summary:** A computer science student with expertise in artificial intelligence and machine learning, seeking an internship to apply technical skills.
- **Skills:** C++, Java, Python, JavaScript, Scala, Rust, GitHub, Linux, SQL Server, MySQL, SOLID Principles, Design Patterns

## 2. Job Fit
- **Fit score:** 85 / 100
- **Matching skills:** C++, Java, Python, JavaScript, Scala, Rust, GitHub, Linux, SQL Server, MySQL, SOLID Principles, Design Patterns
- **Missing skills:** none
- **Notes:** The candidate has strong technical skills relevant to the software engineering role. They have experience with multiple programming languages and tools commonly used in the field.

## 3. Feedback
**Strengths**
- The candidate demonstrates proficiency in several key programming languages and technologies that align well with the requirements of the software engineering position.
- Their educational background includes a BSc in Computer and Information Science, which provides a solid foundation in computer science principles and practices.

**Improvements**
- While the candidate's skill set is impressive, there is room for further specialization in specific areas such as cloud computing or cybersecurity.

**Skills worth learning**
- Cloud Computing (AWS, Azure, Google Cloud)
- Cybersecurity Fundamentals
- Advanced Machine Learning Techniques

## 4. Cover Letter
Dear Hiring Manager,

I am excited to apply for your position as a Software Engineer at [Company Name]. With my strong background in programming languages such as C++, Java, Python, JavaScript, and Rust, I bring a unique set of skills that align perfectly with your requirements.

My academic journey at the University of Ain Shams has equipped me with comprehensive knowledge across various domains including database management systems like SQL Server and MySQL, software design patterns, and modern development practices such as SOLID principles and design patterns. Additionally, my hands-on experience in projects like Online Games Popularity Analysis, CyberGuard AI, Mini Wasalny, and ASH Shell demonstrates my ability to work effectively in both individual and collaborative environments.

I am particularly drawn to your focus on Python and its integration with other technologies. My proficiency in these areas would enable me to contribute significantly to your team by leveraging existing tools and frameworks while also contributing innovative solutions to complex problems. Your emphasis on RESTful APIs and cloud computing services is another area where I can make valuable contributions, ensuring efficient data handling and scalable application deployment.

I am eager to bring my passion for technology and dedication to excellence to your organization. I believe my educational background, practical project experience, and continuous skill development will be instrumental in achieving our goals together. Thank you for considering my application. I look forward to the opportunity to discuss how my qualifications can help us succeed.

Best regards,
Roqaya Muhammad

## 5. Workflow Trace
| Agent | Status | Time |
|---|---|---|
| Resume Agent | done | 19.0 s |
| Job-Fit Agent | done | 6.1 s |
| Feedback Agent | done | 7.4 s |
| Cover Letter Agent | done | 15.3 s |

*Model used: Qwen2.5-1.5B-Instruct*



Local LLM: Qwen2.5-1.5B-Instruct
Loaded   : yes


## 16. Save Final Report

In [27]:
if "result" in globals() and result.get("report_path"):
    print("Saved:", os.path.abspath(result["report_path"]))
    try:
        from google.colab import files
        files.download(result["report_path"])
    except Exception:
        print("(Download is available in Colab: use the file browser on the left.)")
else:
    print("No report yet — run section 14 first.")

Saved: /kaggle/working/career_report.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 17. Streamlit Demo
The Streamlit interface is **defined inside this notebook** (no separate `app.py` file). The launcher cell after it assembles a temporary script at runtime from:
1. the notebook's own `# @core:...` cells (`LLMManager`, agents, `Orchestrator`, etc.) — **no logic is duplicated**, and
2. the UI code below.

UI contents: resume upload + job-description box · **Run Career Coach** button · workflow status (one line per agent) · tabs *Profile / Job Fit / Feedback / Cover Letter / System Status* · sidebar switch **Force Primary Model Failure** · download button for `career_report.md`.

**How to launch:** set `LAUNCH_STREAMLIT = True` in the launcher cell and run it. It frees the notebook's GPU memory first (so Streamlit can load its own model copy), starts Streamlit, and opens a tunnel link so you can open it in your browser.

In [28]:
STREAMLIT_UI_CODE = r'''
# ======================= Streamlit UI (uses the notebook's core classes above) =======================
st.set_page_config(page_title="AI Career Coach", page_icon="💼", layout="wide")
st.title("💼 AI Career Coach")
st.caption("Multi-agent system · local Qwen2.5-Instruct models · no API keys")


@st.cache_resource(show_spinner=False)
def get_llm():
    hw = detect_hardware()
    primary = select_models(hw)
    return LLMManager(primary, verbose=True), hw


llm, hw = get_llm()

with st.sidebar:
    st.header("⚙️ Settings")
    st.markdown(f"**Device:** {hw['gpu_name'] or 'CPU'}" + (f" ({hw['vram_gb']} GB)" if hw["cuda_available"] else ""))
    st.markdown(f"**Model:** {llm.spec['label']}")
    if not hw["cuda_available"]:
        st.warning("No GPU detected — generation will be slow.")

uploaded = st.file_uploader("Upload Resume (PDF)", type=["pdf"])
job_description_ui = st.text_area("Paste the job description", height=180,
                                   placeholder="e.g. Software Engineer, Python, 2+ years, REST APIs, SQL, cloud...")
run_clicked = st.button("Run Career Coach", type="primary",
                        disabled=uploaded is None or not job_description_ui.strip())

if run_clicked and uploaded is not None:
    pdf_file = os.path.join(tempfile.gettempdir(), "uploaded_resume.pdf")
    with open(pdf_file, "wb") as f:
        f.write(uploaded.getbuffer())

    st.subheader("Workflow Status")
    slots = {name: st.empty() for name in AGENT_NAMES}
    for name in AGENT_NAMES:
        slots[name].markdown(f"⏺ {name} — waiting")

    with st.spinner("Running the multi-agent workflow..."):
        orchestrator = Orchestrator(llm)
        result = orchestrator.run(pdf_file, job_description_ui, report_path="career_report.md")

    for name in AGENT_NAMES:
        w = next((x for x in result["workflow"] if x["agent"] == name), None)
        if w is None:
            slots[name].markdown(f"➖ {name} — not reached")
        elif w["status"] == "done":
            slots[name].markdown(f"✓ **{name}** done ({w['seconds']}s)")
        else:
            slots[name].markdown(f"✗ **{name}** {w['status']}")

    if result["status"] == "completed":
        p, fit, fb = result["profile"], result["fit"], result["feedback"]
        tabs = st.tabs(["Profile", "Job Fit", "Feedback", "Cover Letter", "System Status"])

        with tabs[0]:
            st.markdown(f"**Name:** {p['name']}")
            st.markdown(f"**Summary:** {p['summary']}")
            st.markdown("**Skills:** " + ", ".join(p["skills"]))
            if p["experience"]:
                st.markdown("**Experience:**")
                for e in p["experience"]:
                    st.markdown(f"- {e['title']} — {e['company']} ({e['years']})")

        with tabs[1]:
            st.metric("Fit score", f"{fit['fit_score']} / 100")
            st.markdown("**Matching skills:** " + (", ".join(fit["matching_skills"]) or "none detected"))
            st.markdown("**Missing skills:** " + (", ".join(fit["missing_skills"]) or "none"))
            st.markdown(fit["notes"])

        with tabs[2]:
            st.markdown("**Strengths**")
            for s in fb["strengths"]:
                st.markdown(f"- {s}")
            st.markdown("**Improvements**")
            for s in fb["improvements"]:
                st.markdown(f"- {s}")
            if fb["suggested_skills_to_learn"]:
                st.markdown("**Skills worth learning**")
                for s in fb["suggested_skills_to_learn"]:
                    st.markdown(f"- {s}")

        with tabs[3]:
            st.text_area("Cover letter", result["cover_letter"], height=300)

        with tabs[4]:
            st.json(result["llm_status"])

        with open(result["report_path"], "rb") as f:
            st.download_button("Download career_report.md", f, file_name="career_report.md")
    else:
        st.error(result.get("error", "Workflow failed."))
'''

In [29]:
CORE_TAGS = ["imports", "hardware", "json", "llm", "pdf_utils",
             "resume_agent", "fit_agent", "feedback_agent", "letter_agent", "orchestrator"]


def build_streamlit_app(path: str = None) -> str:
    """Assemble the Streamlit script at runtime from the notebook's own core cells + the UI code."""
    ns = get_ipython().user_ns
    history = ns.get("In") or ns.get("_ih") or []
    cells = {}
    for src in history:
        m = re.match(r"\s*# @core:(\w+)", src)
        if m:
            cells[m.group(1)] = src  # latest executed version of each core cell
    missing = [t for t in CORE_TAGS if t not in cells]
    if missing:
        raise RuntimeError(f"Run these notebook cells first (core code missing): {missing}")
    script = "import streamlit as st\nimport tempfile\n\n" + "\n\n".join(cells[t] for t in CORE_TAGS) + "\n\n" + STREAMLIT_UI_CODE
    compile(script, "streamlit_app.py", "exec")  # fail early on syntax errors
    path = path or ("/content/streamlit_app.py" if os.path.isdir("/content") else "streamlit_app.py")
    with open(path, "w", encoding="utf-8") as f:
        f.write(script)
    return path


APP_PATH = build_streamlit_app()
print("Streamlit script generated at runtime:", APP_PATH, "(temporary file, not a project source file)")

Streamlit script generated at runtime: /content/streamlit_app.py (temporary file, not a project source file)


In [30]:
LAUNCH_STREAMLIT = True  # <-- set to True, then run this cell to start the demo UI

if LAUNCH_STREAMLIT:
    import subprocess, os, time, re

    llm.unload()  # free GPU memory: Streamlit loads its own model copy

    log_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content" if os.path.isdir("/content") else "."
    log_path = os.path.join(log_dir, "streamlit.log")

    subprocess.Popen(["streamlit", "run", APP_PATH, "--server.port", "8501", "--server.headless", "true",
                      "--browser.gatherUsageStats", "false"],
                     stdout=open(log_path, "w"), stderr=subprocess.STDOUT)
    print(f"Streamlit log: {log_path}")
    time.sleep(6)

    # cloudflared: a free tunnel with no account needed, more reliable than localtunnel
    # for apps that load several static JS/CSS files (like Streamlit).
    cloudflared_path = os.path.join(log_dir, "cloudflared")
    if not os.path.exists(cloudflared_path):
        subprocess.run(
            ["wget", "-q", "-O", cloudflared_path,
             "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
            check=True
        )
        os.chmod(cloudflared_path, 0o755)

    tunnel_log = os.path.join(log_dir, "cloudflared.log")
    subprocess.Popen([cloudflared_path, "tunnel", "--url", "http://localhost:8501"],
                     stdout=open(tunnel_log, "w"), stderr=subprocess.STDOUT)

    print("Waiting for the tunnel URL...")
    url = None
    for _ in range(30):
        time.sleep(1)
        if os.path.exists(tunnel_log):
            text = open(tunnel_log).read()
            match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", text)
            if match:
                url = match.group(0)
                break

    if url:
        print(f"Open this URL (no password needed): {url}")
    else:
        print(f"Couldn't find the tunnel URL yet — check {tunnel_log} manually for the https://...trycloudflare.com link.")
else:
    print("Set LAUNCH_STREAMLIT = True and re-run this cell to start the Streamlit demo.")
    print("(Locally, you can also save the generated script and run:  streamlit run streamlit_app.py)")


Model unloaded, GPU memory released.
Streamlit log: /kaggle/working/streamlit.log
Waiting for the tunnel URL...
Open this URL (no password needed): https://donate-catalog-conclusions-unit.trycloudflare.com


In [ ]:
!python /kaggle/input/datasets/roqayamuhammad/ai-career-coach/app.py

## 18. Conclusion

**What was built:** a complete multi-agent career coach in one notebook — a Resume Agent (PDF → structured profile), a Job-Fit Agent (profile + job → score and gaps), a Feedback Agent (strengths/improvements), a Cover Letter Agent (tailored letter), and an Orchestrator that connects them, all powered by **local Qwen2.5-Instruct** models with **no API keys**.

**Requirements covered**

| Requirement | Where |
|---|---|
| Data retrieval / processing | `load_resume_text`, `clean_text` |
| LLM analysis | ResumeAgent, JobFitAgent, FeedbackAgent, CoverLetterAgent via `LLMManager` |
| Multiple agents that exchange data | Orchestrator: profile → fit → feedback / cover letter |
| Error handling | PDF validation, empty job description, safe JSON parsing with retries, readable errors |
| Automated output | `career_report.md` |

**Limitations & ideas:** scanned resumes need OCR (not included); small local models can still misjudge fit, so the score should be read as a rough signal, not ground truth; possible extensions — multi-job comparison, LinkedIn-profile input instead of only PDF.

**Troubleshooting**
- *Out of memory / very slow* → Runtime → Change runtime type → T4 GPU, then restart and re-run.
- *"No readable text"* → the PDF is probably a scanned image, not real text.